# Normative Modelling: A Hands-On Tutorial
### From first principles to PCNtoolkit (≈ 2 hours)

**Audience:** graduate students and researchers comfortable with basic Python and regression.

**By the end of this session you will be able to:**
1. Explain why normative modelling complements (and differs from) case–control analysis.
2. Build a normative model from scratch: non-linear mean, site effects, and age-varying variance.
3. Evaluate a normative model properly (explained variance, MSLL, calibration of z-scores).
4. Fit and evaluate warped Bayesian linear regression models with **PCNtoolkit 1.x**.
5. Map individual deviations in a clinical group and interpret heterogeneity.

| Time | Section |
|---|---|
| 0:00 – 0:10 | 0. Setup · 1. Why normative modelling? |
| 0:10 – 0:20 | 2. The dataset |
| 0:20 – 1:00 | 3. Building a normative model from scratch |
| 1:00 – 1:10 | ☕ Break |
| 1:10 – 1:35 | 4. Normative modelling with PCNtoolkit |
| 1:35 – 1:52 | 5. Clinical application: mapping individual deviations |
| 1:52 – 2:00 | 6. Pitfalls, discussion & wrap-up |

> **Data:** everything runs on a *simulated* multi-site cortical thickness dataset, so no downloads or data-use agreements are needed, and — unusually for real data — we know the ground truth.

## 0. Setup

Follow [INSTALLATION.md](INSTALLATION.md) to create the Python environment and select it as this notebook's kernel. Then run the cell below to check that everything works.

Keep these files together in one folder:
- `normative_modelling_tutorial.ipynb`: this notebook,
- `simulate_data.py`: creates the simulated data set,
- `plotting.py`: a few plotting helpers,
- `solutions.ipynb`: solutions to the exercises.

In [35]:
import sys
from importlib.metadata import version
import numpy, scipy, pandas, matplotlib, sklearn
from pcntoolkit import BLR, BsplineBasisFunction, NormData, NormativeModel

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PCNtoolkit version:", version("pcntoolkit"))


Python executable: /Users/johannabayer/anaconda3/envs/normative/bin/python
Python version: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:38:15) [Clang 20.1.8 ]
PCNtoolkit version: 1.3.0


The executable path should point into `normative` or `.venv`, and PCNtoolkit should report a 1.x version. If you get an error, see **Troubleshooting** in [INSTALLATION.md](INSTALLATION.md).

In [ ]:
# ---------------------------------------------------------------------------
# Imports and global settings
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")          # keep the notebook output readable during teaching

import numpy as np                         # arrays and linear algebra
import pandas as pd                        # tabular data (one row per subject)
import matplotlib.pyplot as plt            # plotting
from scipy import stats, optimize          # distributions/tests; numerical optimisation (Section 3.4)
from sklearn.preprocessing import SplineTransformer        # B-spline basis for non-linear age effects

# Plotting helpers for Sections 2 and 5 (in plotting.py, next to this notebook)
from plotting import plot_dataset, plot_patient_heatmap, plot_extreme_counts, plot_extreme_by_region

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Why normative modelling? (≈ 10 min)

**The case–control paradigm** asks: *does the average patient differ from the average control?*
It implicitly assumes patients are a homogeneous group. For most psychiatric and many neurological conditions this is false: two people with the same diagnosis can have very different underlying biology.

<img src="imgs/Screenshot1.png" width="800">

*Bayer, 2025*

<img src="imgs/Screenshot2.png" width="800">

*Ruthreford et al., 2022*

**Normative modelling** borrows the logic of paediatric growth charts:
1. Learn the *distribution* of a brain measure (not just its mean) across a large reference population, as a function of covariates like age, sex and scanner site.
2. Place each new individual on that chart and ask *how atypical are they?*, expressed as a **deviation (z) score**.

$$z_{i} = \frac{y_i - \hat{\mu}(x_i)}{\hat{\sigma}(x_i)}$$

Two things are essential and easy to get wrong:
- $\hat\mu(x)$ must capture **non-linear** trajectories (brain development and ageing are not straight lines).
- $\hat\sigma(x)$ must capture how **variability** changes with covariates — otherwise z-scores are miscalibrated, and "extreme" means different things at different ages.

### Normative modelling is a two-step procedure

**Step 1: Fit (train).** Using a **reference cohort of healthy controls** only, learn how the brain measure $y$ is distributed as a function of the covariates $x$: its expected value $\hat\mu(x)$ *and* its spread $\hat\sigma(x)$. This is the growth chart.

**Step 2: Apply (test).** Take a *new* person who was **not** used for fitting. Put their covariates into the fitted model to get their expected value and spread, and compare it with their actual measurement to get a z-score.

**In short:**
- **Fit:** healthy controls (the training set). We use their covariates *and* brain measures to learn the expected value and the spread. The result is a fitted model.
- **Apply:** new people (held-out controls and patients). We use their covariates *and* brain measures to compute a z-score for each person and region.

Two rules follow from this:
- **Patients are never used for fitting.** The model describes the healthy range; patients are only *compared* against it. If patients were in the reference cohort, their deviations would partly be absorbed into what counts as "normal".
- **Evaluate on held-out controls.** Controls who were not used for fitting should get z-scores with mean ≈ 0 and SD ≈ 1, and about 5% should fall beyond ±1.96. Checked on the training data, a model always looks better than it really is.

In this tutorial: `train` (70% of controls) is used for fitting; `test` (the other 30%) and `patients` are used for applying.

### What goes into the model: responses, covariates and site effects

- **Response variable $y$:** the brain measure we want a norm for, e.g. cortical thickness of one region. We fit one separate model per region (or per vertex or voxel).
- **Covariates $x$:** variables that explain *healthy* differences between people, and that the norm should take into account. **Age** is the main one (its effect is non-linear), and **sex** is usually included. A z-score then means "unusual *for someone of this age and sex*". Choose covariates carefully: don't include variables that are a *consequence* of the condition you study (e.g. total brain volume in an atrophy disorder), or you'll remove the effect you're looking for.
- **Site (batch) effects:** data from different scanners, protocols or cohorts differ for technical reasons, not biological ones. A site can **shift** the values (e.g. all thickness values 0.05 mm higher) and change their **spread** (a noisier scanner). If we ignore site, a person from a "high" site looks thicker than they are, and a noisy site produces too many "extreme" z-scores. So site goes into the model for both the mean and the variance.
- **Site and age are often confounded.** Many studies recruit a limited age range per site (e.g. a paediatric cohort and an ageing cohort), so age and site effects have to be estimated *together*, which requires data from many sites that overlap in age.

In PCNtoolkit's terms, continuous predictors such as age are called **covariates**, and categorical grouping variables such as site and sex are called **batch effects**.

> **Discussion prompt (2 min):** A treatment works for 20% of patients who share an abnormality in region X, while the other 80% have abnormalities elsewhere. What would a case–control study of region X show?

## 2. The dataset (≈ 10 min)

We simulate cortical thickness (mm) in **8 regions** for **3,000 healthy controls** from **4 scanner sites** with different age ranges, plus **240 patients**. The simulation deliberately includes the things that make real data hard:

- a **non-linear** lifespan trajectory (early thinning, then slower decline),
- **heteroscedasticity** — variance grows with age,
- **site effects** on both location and scale,
- **skewed** noise in some regions,
- **heterogeneous** patients: each has abnormal thinning in a *different* 1–3 regions (and some have none).

The simulation code is in `simulate_data.py`, next to this notebook, so here we just load the result. You don't need to read that file, but it's there if you want to see how the data are made. We get:

- `train`: 70% of the controls, used to **fit** the models,
- `test`: the other 30% of the controls, held out to **evaluate** the models,
- `patients`: the 240 patients,
- `truth`: a patients × regions table saying who is *truly* abnormal where (a luxury we never have with real data),
- `ROIS` (the 8 region names), `SITES`, `N_ROI` (number of regions) and `n_pat` (number of patients).

In [ ]:
from simulate_data import ROIS, SITES, N_ROI, n_pat, train, test, patients, truth

print("train controls:", len(train), "| test controls:", len(test), "| patients:", len(patients))
train.head()

In [55]:
train.age[train.sex ==0]

0       52.942094
2       18.132371
3       37.517272
5       67.724563
7       38.618961
          ...    
2094    15.634623
2095    37.673969
2096    73.659342
2097    13.632014
2098    14.125578
Name: age, Length: 1026, dtype: float64

In [ ]:
# Four panels: (1) age coverage of each site, (2) a region with symmetric noise,
# (3) a region with strongly skewed noise, (4) another skewed region. Colours = sites.
plot_dataset(train, SITES)

**Things to notice:**
- Sites cover different age ranges, so **site and age are confounded**. A model that ignores site will partly misattribute site differences to age.
- The spread of the data increases with age.
- The entorhinal cloud has a long upper tail — a Gaussian model will struggle there.

### A quick case–control analysis (the "traditional" view)
Let's compare patients with age-matched controls, region by region.

In [ ]:
# Classic case-control comparison, one region at a time.
# We restrict controls to adults because all patients are >= 18 years old.
adult_ctrl = test[test.age >= 18]
res = []
for roi in ROIS:
    t, p = stats.ttest_ind(patients[roi], adult_ctrl[roi])          # two-sample t-test
    # Cohen's d: mean difference in units of the control SD (negative = patients thinner)
    d = (patients[roi].mean() - adult_ctrl[roi].mean()) / adult_ctrl[roi].std()
    res.append({"roi": roi, "cohen_d": round(d, 2), "p": p,
                # ground truth: what fraction of patients is actually affected in this region?
                "% patients truly abnormal here": round(100 * truth[:, ROIS.index(roi)].mean(), 1)})
pd.DataFrame(res).set_index("roi")

,cohen_d,p,% patients truly abnormal here
roi,,,
frontal_pole,-0.44,2.048296e-07,8.3
precentral,-0.49,1.060911e-09,7.9
sup_temporal,-0.64,5.273801e-14,11.2
insula,-0.78,2.222979e-18,20.8
precuneus,-0.49,5.312625e-09,8.8
lat_occipital,-0.55,2.753921e-11,10.8
entorhinal,-0.69,3.121413e-16,21.2
cingulate,-0.39,2.443358e-06,9.2


Effect sizes are modest, yet we *know* that in every region a subset of patients is thinner by 3–4.5 SD. The group statistic cannot distinguish "every patient is a little thinner" from "a few patients are much thinner", and those two scenarios have very different clinical implications. Also note this crude comparison ignores age, sex and site differences between groups. Keep this table in mind; we'll revisit it in Section 5.

### ✏️ Exercise 2a (3 min)
For `entorhinal`, split the patients into those who are **truly abnormal** in that region and those who are not (use `truth[:, j]` with `j = ROIS.index("entorhinal")`). Compute Cohen's d against the adult controls for each group separately, and for all patients together. How does the "all patients" effect size relate to the other two?

In [ ]:
# Exercise 2a — your code here
# Tip: patients[truth[:, j]] selects the truly abnormal patients; patients[~truth[:, j]] the rest.

## 3. Building a normative model from scratch (≈ 40 min)

We'll build the model up in stages and evaluate each one. We focus on **one skewed region (`entorhinal`)** and **one symmetric region (`precuneus`)**, then scale up to all regions.

### 3.1 How do we know if a normative model is good?

A normative model makes a *probabilistic* prediction, so accuracy of the mean is not enough. We'll report:

| Metric | What it measures | Good value |
|---|---|---|
| **EV** (explained variance) | how well the mean fits | higher |
| **MSLL** (mean standardised log loss) | quality of the full predictive distribution, relative to a trivial model that predicts the training mean & variance | more negative |
| **z mean / z sd** | calibration of location / scale | 0 / 1 |
| **skew, excess kurtosis of z** | shape calibration (tails!) | 0 / 0 |
| **% \|z\| > 1.96** | tail calibration | ≈ 5% |

The last three matter most for clinical use: if the model's tails are wrong, "extreme deviations" are wrong.

In [ ]:
def evaluate(y, mu, sd, y_train):
    # Compare observed values y with the model's predicted mean (mu) and SD (sd).
    # y_train is only needed for the "trivial" model used by MSLL.
    z = (y - mu) / sd                                         # deviation scores

    # Log-likelihood of each person under our model...
    ll_model = stats.norm.logpdf(y, mu, sd)
    # ...and under a trivial model that predicts the training mean and SD for everybody.
    ll_trivial = stats.norm.logpdf(y, y_train.mean(), y_train.std())

    results = {}
    results["EV"] = 1 - np.var(y - mu) / np.var(y)            # higher = better mean fit
    results["MSLL"] = np.mean(ll_trivial - ll_model)          # more negative = better
    results["z_mean"] = np.mean(z)                            # should be ~0
    results["z_sd"] = np.std(z)                               # should be ~1
    results["z_skew"] = stats.skew(z)                         # should be ~0 (symmetric)
    results["z_exkurt"] = stats.kurtosis(z)                   # should be ~0 (Gaussian tails)
    results["% |z|>1.96"] = 100 * np.mean(np.abs(z) > 1.96)   # should be ~5
    return pd.Series(results).round(3)


def calibration_by_age(z, age):
    # Percentage of people with |z| > 1.96 in each age group.
    # A well-calibrated model gives ~5% in EVERY age group, not just on average.
    age_group = pd.cut(age, bins=[6, 20, 40, 60, 88])
    is_extreme = pd.Series(np.abs(z) > 1.96)
    return (100 * is_extreme.groupby(age_group, observed=True).mean()).round(1)

### 3.2. Four stages of normative modelling


Each stage fixes one specific failure of the stage before it. The pattern to watch for is that a model can predict the *mean* well and still give badly wrong z-scores.

| Stage | Model | What it adds | What still goes wrong |
|---|---|---|---|
| **1. Naive** (3.2) | straight line in age, one SD for everyone | the basic idea: predict, then standardise the residual | trajectories are not linear; variability changes with age |
| **2. Non-linear mean** (3.3) | B-spline basis of age + sex + site | flexible lifespan curves and site/sex offsets for the **mean** | still one SD for everyone, so z-scores are miscalibrated across ages and sites |
| **3. Heteroscedastic** (3.4) | mean *and* log-SD both depend on covariates | a **variance** that changes with age and site | assumes Gaussian noise; skewed or heavy-tailed regions get the wrong tails |
| **4. Warped likelihood** (Section 4) | warped Bayesian linear regression in PCNtoolkit | a learned **shape** (skew and kurtosis) via a sinh-arcsinh transform | — (calibrated location, scale *and* shape) |

**The four models as equations.** $y$ is the brain measure (e.g. cortical thickness) and $x$ holds a person's covariates (age, sex, site).

**Stage 1: straight line, one SD for everyone**
$$y = \beta_0 + \beta_1\,\text{age} + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\ \sigma^2)$$

**Stage 2: non-linear mean via a spline expansion $\phi(x)$**
$$y = \phi(x)^\top \beta + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\ \sigma^2)$$
$$\phi(x) = \big[\,1,\ B_1(\text{age}),\ \dots,\ B_K(\text{age}),\ \text{sex},\ \text{site}_B,\ \text{site}_C,\ \text{site}_D\,\big]$$

$\phi(x)$ expands the covariates into many columns: each $B_k$ is one smooth spline "bump" over age, so the weighted sum $\phi(x)^\top\beta$ can follow a curve. The model is still linear in $\beta$, so it's still ordinary regression. $\phi(x)$ is exactly one row of the design matrix built by `make_design()`.

**Stage 3: the SD also depends on the covariates**
$$y \sim \mathcal{N}\big(\phi_\mu(x)^\top\beta,\ \sigma(x)^2\big), \qquad \log\sigma(x) = \phi_\sigma(x)^\top\gamma$$

$\phi_\mu(x)$ is the same expansion as in stage 2. $\phi_\sigma(x)$ uses the same age splines and site columns, but no sex column.

**Stage 4: warped likelihood**
$$f(y) \sim \mathcal{N}\big(\phi_\mu(x)^\top\beta,\ \sigma(x)^2\big)$$

$f$ is a sinh-arcsinh warp: a flexible, monotonic transformation of $y$ whose parameters are learned together with $\beta$ and $\gamma$. The data are Gaussian after warping, so in the original units the model can be skewed and heavy-tailed.

In every stage the deviation score is the same idea: $z = \dfrac{y - \hat\mu(x)}{\hat\sigma(x)}$, computed in the warped space for stage 4.

In short, stage 1 is the baseline, and stages 2 to 4 model the **mean**, the **variance** and the **shape** of the distribution in turn. Stages 1 to 3 are built from scratch here so that nothing is a black box; stage 4 uses PCNtoolkit. After every stage we use the same metrics (below) and check calibration *by age*, so you can see which problem each step solves.


### 3.2 Stage 1 — the naive model: linear in age, constant variance

In [ ]:
roi = "precuneus"      # later you will change this to "entorhinal" and re-run 3.2 to 3.4
y_train = train[roi].values
y_test = test[roi].values

# Design matrix = [intercept, age]  -> a straight line in age.
X_train = np.column_stack([np.ones(len(train)), train["age"]])
X_test = np.column_stack([np.ones(len(test)), test["age"]])

# Ordinary least squares for the mean.
# lstsq returns several things; [0] takes the first one, the weights.
beta = np.linalg.lstsq(X_train, y_train, rcond=None)[0]

# ONE residual SD for everybody (constant variance).
residuals = y_train - X_train @ beta
sd_naive = np.std(residuals)

# Predict the held-out controls and compute their z-scores.
mu_test = X_test @ beta
z_naive = (y_test - mu_test) / sd_naive

results_stage1 = evaluate(y_test, mu_test, sd_naive, y_train)
print(results_stage1)
print("\n% |z|>1.96 by age group (target ≈ 5):")
print(calibration_by_age(z_naive, test["age"]))

In [ ]:
grid = np.linspace(6, 88, 200)          # ages at which we draw curves (reused later)
line = beta[0] + beta[1] * grid         # the fitted straight line

plt.figure(figsize=(7, 4))
plt.scatter(test["age"], y_test, s=4, alpha=0.4, c="grey")
plt.plot(grid, line, "k")
# Shaded band = the model's 95% "normal range": mean ± 1.96 × SD (the same SD at every age)
plt.fill_between(grid, line - 1.96 * sd_naive, line + 1.96 * sd_naive, alpha=0.2)
plt.title("Naive model: " + roi)
plt.xlabel("age")
plt.ylabel("thickness (mm)")
plt.show()

The straight line misses childhood thinning, and the constant band is too wide for the young and too narrow for the old. A 70-year-old flagged as "extreme" by this model may be perfectly typical for their age.

### 3.3 Stage 2 — non-linear mean (B-splines) + covariates

We expand age into a **cubic B-spline basis**: a set of smooth, local "bump" functions whose weighted sum can trace any smooth curve. This keeps the model linear in its parameters (so it's still just regression), while allowing non-linear trajectories. We also add **sex** and **site** (one-hot, first site as reference).

$$y = \phi(x)^\top \beta + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0,\ \sigma^2)$$
$$\phi(x) = \big[\,1,\ B_1(\text{age}),\ \dots,\ B_K(\text{age}),\ \text{sex},\ \text{site}_B,\ \text{site}_C,\ \text{site}_D\,\big]$$

$\phi(x)$ expands the covariates into many columns: each $B_k$ is one smooth spline "bump" over age, so the weighted sum $\phi(x)^\top\beta$ can follow a curve. The model is still linear in $\beta$, so it's still ordinary regression. $\phi(x)$ is exactly one row of the design matrix built by `make_design()`.

In [ ]:
# Step 1: learn the spline basis from the TRAINING ages (done once, then reused for all data).
#   n_knots=6              -> how flexible the curve is (more knots = more wiggly)
#   degree=3               -> cubic splines (smooth curves)
#   include_bias=False     -> we add our own intercept column below
#   extrapolation="linear" -> continue as a straight line outside the training ages
spline = SplineTransformer(n_knots=6, degree=3, include_bias=False, extrapolation="linear")
spline.fit(train[["age"]])


# Step 2: a function that builds the design matrix for any data frame.
def make_design(df, spline, use_sex=True, use_site=True):
    columns = []
    columns.append(np.ones((len(df), 1)))            # intercept
    columns.append(spline.transform(df[["age"]]))    # one column per spline "bump"
    if use_sex:
        columns.append(df[["sex"]].values)           # sex (0/1)
    if use_site:
        # One 0/1 column per site. site_A is left out: it is the reference site,
        # and its level is already captured by the intercept.
        for site in ["site_B", "site_C", "site_D"]:
            columns.append((df[["site"]].values == site).astype(float))
    return np.hstack(columns)


X_train = make_design(train, spline)
X_test = make_design(test, spline)
print("design matrix shape:", X_train.shape)   # (subjects, 1 intercept + splines + 1 sex + 3 sites)

# Plot the spline basis: the model's mean curve is a weighted sum of these bumps.
basis = spline.transform(pd.DataFrame({"age": grid}))
plt.figure(figsize=(7, 2.5))
plt.plot(grid, basis)
plt.title("Cubic B-spline basis over age")
plt.show()

In [ ]:
# Same least-squares recipe as Stage 1, just with the richer design matrix.
beta = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
sd_homo = np.std(y_train - X_train @ beta)     # still ONE SD for everybody

mu_test = X_test @ beta
z_spline = (y_test - mu_test) / sd_homo

results_stage2 = evaluate(y_test, mu_test, sd_homo, y_train)
print(results_stage2)
print("\n% |z|>1.96 by age group:")
print(calibration_by_age(z_spline, test["age"]))

The mean fit (EV) improves substantially, but calibration across age is still poor: the variance problem remains. Overall "% |z|>1.96" can look fine while being badly wrong within subgroups — **always check calibration conditionally**, by age, site and sex.

### 3.4 Stage 3 — heteroscedastic model: modelling the variance

We now let *both* the mean and the log standard deviation depend on covariates:

$$y \sim \mathcal{N}\big(\mu(x),\ \sigma(x)^2\big),\qquad \mu(x) = X_\mu\beta,\qquad \log\sigma(x) = X_\sigma\gamma$$

and estimate $\beta, \gamma$ together by maximum likelihood: we write down the negative log-likelihood and let `scipy.optimize.minimize` find the weights that make it as small as possible.

$$\text{NLL} = \sum_i \Big[\log\sigma_i + \tfrac{1}{2}\,r_i^2/\sigma_i^2\Big],\quad r_i = y_i - \mu_i$$

In [ ]:
# The model:  y ~ Normal( mean = X_mean @ beta,  SD = exp(X_var @ gamma) )
# Taking exp() of X_var @ gamma guarantees that the predicted SD is always positive.


def neg_log_likelihood(params, y, X_mean, X_var):
    # params holds all the weights: first the mean weights (beta), then the log-SD weights (gamma).
    n_mean = X_mean.shape[1]
    beta = params[:n_mean]
    gamma = params[n_mean:]
    mu = X_mean @ beta               # predicted mean for every person
    sd = np.exp(X_var @ gamma)       # predicted SD for every person
    return -np.sum(stats.norm.logpdf(y, mu, sd))


def fit_normative_model(df, y, n_knots=6, site_in_mean=True, site_in_var=True):
    # 1. Spline basis for age (the same one is used for the mean and the SD).
    spline = SplineTransformer(n_knots=n_knots, degree=3, include_bias=False, extrapolation="linear")
    spline.fit(df[["age"]])

    # 2. Design matrices. The SD model uses age (and site), but not sex.
    X_mean = make_design(df, spline, use_site=site_in_mean)
    X_var = make_design(df, spline, use_sex=False, use_site=site_in_var)

    # 3. Starting values: least squares for the mean, and the same SD for everybody.
    beta_start = np.linalg.lstsq(X_mean, y, rcond=None)[0]
    gamma_start = np.zeros(X_var.shape[1])
    gamma_start[0] = np.log(np.std(y - X_mean @ beta_start))
    start = np.concatenate([beta_start, gamma_start])

    # 4. Let scipy search for the weights with the smallest negative log-likelihood.
    result = optimize.minimize(neg_log_likelihood, start, args=(y, X_mean, X_var))

    # 5. Keep everything we need to make predictions later.
    n_mean = X_mean.shape[1]
    model = {
        "spline": spline,
        "site_in_mean": site_in_mean,
        "site_in_var": site_in_var,
        "beta": result.x[:n_mean],
        "gamma": result.x[n_mean:],
    }
    return model


def predict_normative_model(model, df):
    # Predicted mean and SD for every row of df.
    X_mean = make_design(df, model["spline"], use_site=model["site_in_mean"])
    X_var = make_design(df, model["spline"], use_sex=False, use_site=model["site_in_var"])
    mu = X_mean @ model["beta"]
    sd = np.exp(X_var @ model["gamma"])
    return mu, sd


model = fit_normative_model(train, y_train)
mu_test, sd_test = predict_normative_model(model, test)
z_hetero = (y_test - mu_test) / sd_test

results_stage3 = evaluate(y_test, mu_test, sd_test, y_train)
print(results_stage3)
print("\n% |z|>1.96 by age group:")
print(calibration_by_age(z_hetero, test["age"]))

In [ ]:
# The three stages side by side, for the region chosen in the Stage 1 cell
print("Region:", roi)
pd.DataFrame([results_stage1, results_stage2, results_stage3],
             index=["1 naive", "2 spline", "3 hetero"])

✏️ **Try it (3 min):** go back to the Stage 1 cell, change `roi` to `"entorhinal"`, and re-run the cells from there down to this table. Compare the two tables.

For `precuneus` the heteroscedastic Gaussian model is well calibrated. For `entorhinal`, however, the z-scores remain **skewed with heavy tails** — no amount of mean/variance modelling fixes a *shape* problem. Let's look.

In [ ]:
# Q-Q plot: sorted test z-scores against the quantiles of a standard normal.
# Points on the diagonal = well-calibrated; bending in the tails = wrong shape.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, roi in zip(axes, ["precuneus", "entorhinal"]):
    model = fit_normative_model(train, train[roi].values)
    mu, sd = predict_normative_model(model, test)
    z = (test[roi].values - mu) / sd
    stats.probplot(z, dist="norm", plot=ax)
    ax.set_title("Q–Q plot of test z-scores: " + roi)

plt.tight_layout()
plt.show()

The entorhinal Q–Q plot bends away from the line in the tails. Consequence: in the upper tail we'll produce *too many* "extreme" positive deviations, and in the lower tail the z-scores are compressed. Since clinical interest is usually in the tails, this matters. Section 4 fixes this with **likelihood warping** (a sinh-arcsinh transformation), which lets the model learn skew and kurtosis.

### 3.5 Centile curves ("brain charts")

The fitted model gives us the full predictive distribution at any age, so we can draw growth-chart-style centiles for any site and sex.

In [ ]:
def plot_centiles(model, roi, ax, site="site_D", sex=0):
    # Made-up people: one for every age on the grid, all with the same site and sex.
    fake_people = pd.DataFrame({"age": grid, "sex": sex, "site": site})
    mu, sd = predict_normative_model(model, fake_people)

    # Real test subjects from the same site and sex, as grey dots.
    same_group = test[(test["site"] == site) & (test["sex"] == sex)]
    ax.scatter(same_group["age"], same_group[roi], s=6, alpha=0.5, c="grey")

    # For a Gaussian model, the q-th centile is  mu + sd * (q-th quantile of a standard normal).
    ax.plot(grid, mu + sd * stats.norm.ppf(0.025), "k:", lw=1)    # 2.5th centile
    ax.plot(grid, mu + sd * stats.norm.ppf(0.25), "k--", lw=1)    # 25th centile
    ax.plot(grid, mu, "k-", lw=1)                                 # 50th centile (median)
    ax.plot(grid, mu + sd * stats.norm.ppf(0.75), "k--", lw=1)    # 75th centile
    ax.plot(grid, mu + sd * stats.norm.ppf(0.975), "k:", lw=1)    # 97.5th centile

    ax.set_title(roi + ": " + site + ", sex=" + str(sex))
    ax.set_xlabel("age")
    ax.set_ylabel("thickness (mm)")


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, roi in zip(axes, ["precuneus", "entorhinal"]):
    model = fit_normative_model(train, train[roi].values)
    plot_centiles(model, roi, ax)
plt.tight_layout()
plt.show()

### 3.6 Scaling up: one model per region

In practice we fit an independent model for every brain measure (region, vertex, or voxel). We'll keep the z-scores for test controls and patients for Section 5.

In [ ]:
# Fit one separate normative model per region.
# IMPORTANT: models are trained on healthy controls only. Patients are never
# used for fitting; they are only compared against the norm.
Z_test = np.zeros((len(test), N_ROI))      # z-scores of held-out controls  (subjects x regions)
Z_pat = np.zeros((n_pat, N_ROI))           # z-scores of patients           (subjects x regions)
metrics = {}                               # evaluation metrics per region

for j, roi in enumerate(ROIS):
    model = fit_normative_model(train, train[roi].values)

    # held-out controls
    mu, sd = predict_normative_model(model, test)
    Z_test[:, j] = (test[roi].values - mu) / sd
    metrics[roi] = evaluate(test[roi].values, mu, sd, train[roi].values)

    # patients
    mu, sd = predict_normative_model(model, patients)
    Z_pat[:, j] = (patients[roi].values - mu) / sd

metrics_scratch = pd.DataFrame(metrics).T
metrics_scratch

### ✏️ Exercises (pick one, ~5 min)

**3a. Flexibility vs. overfitting.** Refit `precuneus` with `n_knots` = 3, 6, 15, 30. Plot test MSLL against the number of knots. Where does it stop improving?

**3b. Ignore the site.** Fit a model with no site terms (`site_in_mean=False, site_in_var=False`). Compute `% |z|>1.96` **per site** on the test set. What happens, and why is this especially dangerous when site and diagnosis are correlated?

**3c. Variance model.** Set `site_in_var=False`. Which calibration metric changes, and for which sites?

**3d. One person on the chart.** Fit a model for `entorhinal` and take the first patient (`patients.iloc[[0]]`). What is their expected thickness and SD, their z-score, and their centile (`100 * stats.norm.cdf(z)`)? Is this patient truly abnormal in the entorhinal cortex (`truth[0, j]`)?

**3e. Where does the chart come from?** Use `plot_centiles` to draw the `precuneus` centiles for `site_A`, `site_B` and `site_C`. Each site only recruited a limited age range (see Section 2). Which parts of each chart are backed by data, and which are extrapolation?

Solutions are in `solutions.ipynb`.

In [ ]:
# Exercise 3a / 3b / 3c / 3d / 3e — your code here
# Tip: fit_normative_model(train, y, n_knots=..., site_in_mean=..., site_in_var=...)
# Tip for 3e: plot_centiles(model, roi, ax, site="site_A")

---
## ☕ Break (10 min)
---

## 4. Normative modelling with PCNtoolkit (≈ 25 min)

[PCNtoolkit](https://github.com/predictive-clinical-neuroscience/PCNtoolkit) (v1.x) is organised around three objects:

| Object | Role |
|---|---|
| `NormData` | an `xarray`-based container holding covariates (`X`), responses (`Y`), batch effects (e.g. site, sex) and, after prediction, `Z`, `centiles`, `logp`, `Yhat` and evaluation `statistics` |
| a regression model: `BLR` or `HBR` | the model *template*, cloned once per response variable |
| `NormativeModel` | the orchestrator: scaling, fitting, predicting, evaluation, plotting, saving, and `harmonize` / `synthesize` / `transfer` / `extend` / `merge` |

We'll use **Bayesian linear regression (BLR)**: fast, and conceptually exactly what we built in Section 3 (a B-spline basis for the mean, a model for the variance, and site/sex effects), but with Bayesian priors on the weights. Then we add the **sinh-arcsinh warp** (Fraza et al., 2021), which maps the data to a space where a Gaussian fits and so lets the model learn skewness and kurtosis.

> `HBR` (hierarchical Bayesian regression, built on PyMC) models site effects as random effects and offers a SHASH likelihood; it's more flexible but much slower, so it's left for further reading.

In [ ]:
import pcntoolkit
from pcntoolkit import BLR                     # Bayesian linear regression (the model template)
from pcntoolkit import BsplineBasisFunction    # B-spline expansion of age, like our make_design()
from pcntoolkit import NormativeModel          # fits and predicts one model per region
from pcntoolkit import NormData                # data container
from pcntoolkit import plot_centiles_advanced  # centile curves
from pcntoolkit import plot_qq                 # Q-Q plots of z-scores

# PCNtoolkit prints a message for every step and every region; turn that off.
pcntoolkit.util.output.Output.set_show_messages(False)

### 4.1 Wrapping our data in `NormData`

We keep our own train/test split so the results are directly comparable with Section 3. Note that **site and sex are passed as batch effects**, not as covariates: PCNtoolkit builds the dummy coding for us.

In [ ]:
def to_normdata(name, df):
    # Put one of our data frames into a PCNtoolkit NormData object.
    return NormData.from_dataframe(
        name=name,
        dataframe=df,
        covariates=["age"],              # continuous predictors -> expanded with splines
        batch_effects=["sex", "site"],   # categorical variables -> PCNtoolkit makes the dummies
        response_vars=ROIS,              # one model is fitted for each of these columns
    )


nd_example = to_normdata("train", train)
print(nd_example.coords)

### 4.2 Configure and fit two models

| | heteroscedastic | site/sex effect on mean | site/sex effect on variance | warp |
|---|---|---|---|---|
| `blr` | ✓ | ✓ | ✓ | – |
| `wblr` | ✓ | ✓ | ✓ | sinh-arcsinh |

`fit_predict` fits on the training set, then computes Z-scores, centiles, log-probabilities and evaluation statistics for both training and test sets, and stores them **inside the `NormData` objects**.

In [ ]:
# ---------------------------------------------------------------------------
# Model 1: Gaussian BLR
# ---------------------------------------------------------------------------
# Each BLR option matches something we built by hand in Section 3.
blr_template = BLR(
    basis_function_mean=BsplineBasisFunction(degree=3, nknots=5),  # = our age splines
    heteroskedastic=True,      # = our SD model: variance changes with age
    fixed_effect=True,         # = our site/sex columns in the mean
    fixed_effect_var=True,     # = site_in_var=True: each site gets its own noise level
)
# NormativeModel fits one copy of the template per region.
# The save... options switch off writing files to disk.
blr = NormativeModel(blr_template, savemodel=False, saveresults=False, saveplots=False)

nd_train_blr = to_normdata("train", train)
nd_test_blr = to_normdata("test", test)

# Fit on train, then predict train and test. The results (Z, centiles, statistics)
# are stored INSIDE nd_train_blr and nd_test_blr.
blr.fit_predict(nd_train_blr, nd_test_blr)

# The z-scores as a (subjects x regions) array, like Z_test in Section 3
Z_blr_test = nd_test_blr.Z.values

# ---------------------------------------------------------------------------
# Model 2: warped BLR  (the same, plus the sinh-arcsinh warp)
# ---------------------------------------------------------------------------
wblr_template = BLR(
    basis_function_mean=BsplineBasisFunction(degree=3, nknots=5),
    heteroskedastic=True,
    fixed_effect=True,
    fixed_effect_var=True,
    warp_name="WarpSinhArcsinh",   # NEW: learn skew and kurtosis
    warp_reparam=True,             # a more stable version of the warp
)
wblr = NormativeModel(wblr_template, savemodel=False, saveresults=False, saveplots=False)

# Fresh copies of the data, so the results of model 1 are not overwritten.
nd_train_w = to_normdata("train", train)
nd_test_w = to_normdata("test", test)
wblr.fit_predict(nd_train_w, nd_test_w)      # slower: the warp is fitted as well
Z_wblr_test = nd_test_w.Z.values

# ---------------------------------------------------------------------------
# Score the patients with the fitted warped model (no refitting!)
# ---------------------------------------------------------------------------
nd_pat = to_normdata("patients", patients)
wblr.predict(nd_pat)
Z_wblr_pat = nd_pat.Z.values

### 4.3 Evaluate

PCNtoolkit computes a standard set of metrics per response variable (EXPV, MSLL, NLL, SMSE, MACE, Shapiro–Wilk W of the Z-scores, …). We also reuse our own calibration check from Section 3.

In [ ]:
# get_statistics_df() gives one row per region and one column per metric.
#   EXPV     : explained variance (higher = better mean fit)
#   MSLL     : mean standardised log loss (more negative = better)
#   ShapiroW : Shapiro-Wilk W of the z-scores (closer to 1 = more Gaussian z-scores)
cols = ["EXPV", "MSLL", "ShapiroW"]
stats_blr = nd_test_blr.get_statistics_df()[cols]
stats_wblr = nd_test_w.get_statistics_df()[cols]
pd.concat({"BLR": stats_blr, "warped BLR": stats_wblr}, axis=1).round(3)

In [ ]:
def z_calibration(Z):
    # Shape checks per region on a (subjects x regions) matrix of z-scores.
    # Ideal values: skew 0, excess kurtosis 0, ~5% beyond ±1.96.
    table = pd.DataFrame(index=ROIS)
    table["z_skew"] = stats.skew(Z)
    table["z_exkurt"] = stats.kurtosis(Z)
    table["% |z|>1.96"] = 100 * np.mean(np.abs(Z) > 1.96, axis=0)
    return table.round(2)


# Compare our from-scratch model with the two PCNtoolkit models on the SAME test subjects.
pd.concat({"scratch (Gaussian, hetero)": z_calibration(Z_test),
           "PCN BLR": z_calibration(Z_blr_test),
           "PCN warped BLR": z_calibration(Z_wblr_test)}, axis=1)

**What to look for:** Gaussian BLR should behave much like our from-scratch model (same ingredients). In the heavily skewed regions (`entorhinal`, `cingulate`, `lat_occipital`) the **warped** model should bring z-score skewness and kurtosis close to 0 and raise Shapiro–Wilk W towards 1, at little cost in the symmetric regions (`frontal_pole`, `precuneus`).

### 4.4 Built-in plots
`plot_centiles_advanced` draws centile curves for a chosen batch (here site D, sex 0) and can **harmonise** the scatter data, i.e. show every subject as if scanned at that site. `plot_qq` shows Q–Q plots of the test Z-scores. Both draw one panel per region.

In [ ]:
plot_centiles_advanced(
    wblr,                                              # the fitted model to draw centiles from
    scatter_data=nd_train_w,                           # which subjects to show as dots
    centiles=[0.025, 0.25, 0.5, 0.75, 0.975],          # same centiles as our own plot in 3.5
    batch_effects={"site": ["site_D"], "sex": ["0"]},  # draw curves for this group (values are strings)
    show_other_data=True,                              # also show subjects from other groups...
    harmonize_data=True,                               # ...shifted to site_D / sex 0 so they're comparable
);

In [ ]:
# Q-Q plot per region of the held-out controls' z-scores under the warped model.
plot_qq(nd_test_w, plot_id_line=True);

### Beyond this session
The same `NormativeModel` object supports:
- `harmonize(data, reference_batch_effect={...})`: remove site effects from the raw data;
- `synthesize(...)`: sample synthetic subjects from the fitted model;
- `transfer(...)` / `extend(...)`: adapt a model to a **new site** using a small sample of its controls (see the *Transfer and extend* tutorial; `HBR` supports transfer);
- `merge(...)` and federated fitting across institutions without sharing raw data;
- `Runner(...)`: fit thousands of models (vertices/voxels) in parallel on SLURM/Torque clusters.

### ✏️ Exercise 4a (5 min)
Refit the warped model with `fixed_effect_var=False`. Compare `% |z|>1.96` **per site** with the full model (hint: `pd.Series(np.abs(Z[:, j]) > 1.96).groupby(test.site).mean()`). Why does site-specific variance matter for clinical use?

### ✏️ Exercise 4b (3 min)
Draw the Q–Q plots for the **Gaussian** BLR model (`nd_test_blr`) and compare them with the warped model's plots in 4.4. In which regions does the warp make the biggest difference? Does that match the `z_skew` column in 4.3?

In [ ]:
# Exercise 4a / 4b — your code here
# Tip for 4a: copy the "Model 2" code, change fixed_effect_var, and take the z-scores with .Z.values
# Tip for 4b: plot_qq(nd_test_blr, plot_id_line=True);

## 5. Clinical application: mapping individual deviations (≈ 17 min)

Now the payoff. We use the z-scores from the warped BLR model. A common convention flags **|z| > 2.6** (≈ p < .01, two-sided) as an *extreme deviation*. Here we focus on negative deviations (thinning).

In [ ]:
# Use the z-scores from the warped BLR model.
Zc = Z_wblr_test     # held-out controls  (subjects x regions)
Zp = Z_wblr_pat      # patients           (subjects x regions)

# "Extreme deviation" = z below -2.6 (one-sided p ≈ 0.005). We look at thinning only.
THR = -2.6
ext_ctrl = Zc < THR      # True where a control is extreme  (subjects x regions)
ext_pat = Zp < THR       # True where a patient is extreme   (subjects x regions)

In [ ]:
# (a) Heatmap of all patients' z-scores: rows = regions, columns = patients.
# Red = thinner than expected, blue = thicker.

# Sort the patients so those with the most extreme regions come first.
n_extreme_per_patient = ext_pat.sum(axis=1)
order = np.argsort(-n_extreme_per_patient)     # the minus sign sorts from most to fewest

plot_patient_heatmap(Zp[order], ROIS)

In [ ]:
# (b) In how many regions is each person extreme? Controls vs. patients.
plot_extreme_counts(ext_ctrl, ext_pat)

In [ ]:
# (c) In each region, what % of controls and of patients is extreme?
plot_extreme_by_region(ext_ctrl, ext_pat, ROIS)

**Reading the figures:**
- **(a)** Almost no two patients share the same pattern — the "average patient" doesn't exist.
- **(b)** Many patients have one or more extreme deviations; almost no controls do. This per-person *count* is itself a useful summary measure.
- **(c)** No single region is abnormal in more than a minority of patients, which is why the group-level t-tests in Section 2 looked unimpressive.

Because the data are simulated, we can also check how accurately deviations recover the ground truth.

In [ ]:
# Compare flagged deviations with the ground-truth abnormality mask (patient x region cells).
tp = (ext_pat & truth).sum()     # true positives : flagged AND truly abnormal
fn = (~ext_pat & truth).sum()    # false negatives: missed abnormalities
fp = (ext_pat & ~truth).sum()    # false positives: flagged but actually normal
tn = (~ext_pat & ~truth).sum()   # true negatives
print(f"Sensitivity: {tp / (tp + fn):.2f}   Specificity: {tn / (tn + fp):.3f}")

# In healthy held-out controls every flag is a false positive. For a calibrated
# model, P(z < -2.6) = 0.47%, so that's the rate we expect.
print(f"False-positive rate in held-out controls: {100 * ext_ctrl.mean():.2f}%  (nominal ≈ 0.47%)")

# Group-level statistics are still available, now on covariate-adjusted z-scores.
print("\nGroup-level test on z-scores (patients vs. controls):")
pd.DataFrame({"mean z (patients)": Zp.mean(0).round(2),
              "t-test p": [stats.ttest_ind(Zp[:, j], Zc[:, j]).pvalue for j in range(N_ROI)],
              "% patients extreme": (100 * ext_pat.mean(0)).round(1),
              "% patients truly abnormal": (100 * truth.mean(0)).round(1)}, index=ROIS)

Sensitivity: 0.85   Specificity: 0.996
False-positive rate in held-out controls: 0.29%  (nominal ≈ 0.47%)

Group-level test on z-scores (patients vs. controls):


,mean z (patients),t-test p,% patients extreme,% patients truly abnormal
frontal_pole,-0.25,1.091419e-04,7.9,8.3
precentral,-0.33,7.288569e-05,6.7,7.9
sup_temporal,-0.45,2.502248e-09,9.6,11.2
insula,-0.87,3.493297e-21,18.8,20.8
precuneus,-0.37,2.115701e-06,7.5,8.8
lat_occipital,-0.52,2.149495e-07,9.2,10.8
entorhinal,-0.95,1.927179e-18,18.8,21.2
cingulate,-0.42,9.343554e-06,8.3,9.2


Normative modelling does **not** replace group statistics — you can still test mean z-scores between groups, with the bonus that age, sex and site are already accounted for. It *adds* the individual level.

### ✏️ Exercise 5a (5 min)
Repeat the false-positive analysis using the **non-warped** z-scores (`Z_test` from Section 3). Which regions produce excess false positives, and in which direction? What would this mean for a clinical study of the entorhinal cortex?

### ✏️ Exercise 5c (5 min)
Make a **deviation profile** for one patient: a bar chart of their z-scores across the 8 regions (`Zp[i]`), with a dashed line at −2.6. Start with `i = order[0]`, the patient with the most extreme deviations. Compare the flagged regions with the truly abnormal ones (`truth[i]`). Then try a patient with no extreme deviations. How would you explain this chart to a clinician?

### ✏️ Exercise 5b (open)
Use the patient × region matrix of z-scores as features: cluster patients (e.g. `sklearn.cluster.KMeans`) or predict a clinical variable. What are the risks of interpreting clusters found this way as "subtypes"?

In [ ]:
# Exercise 5a / 5b / 5c — your code here
# Tip for 5b: from sklearn.cluster import KMeans; KMeans(n_clusters=3, n_init=10).fit_predict(Zp)
# Tip for 5c: plt.bar(ROIS, Zp[i]); plt.axhline(-2.6, ls="--")

## 6. Pitfalls, discussion & wrap-up (≈ 8 min)

**Common pitfalls**
1. **The reference cohort defines "normal".** A model trained on healthy, highly educated, mostly white volunteers encodes that population. Deviations are always relative to *who was in the reference sample*.
2. **Unseen sites.** Z-scores from a site not in training can be biased. Include some controls from each new site, and use a model that supports site adaptation or transfer (e.g. HBR).
3. **Extrapolation.** Centiles outside the training age range are guesses; check age coverage per site.
4. **Checking only overall calibration.** Check calibration by age, sex and site — and look at the tails.
5. **Deviation ≠ disease.** An extreme deviation is statistically unusual, not necessarily pathological. Multiple comparisons across thousands of vertices also produce many chance "extremes".
6. **Covariate choice.** Don't include covariates that are *consequences* of the condition (e.g. total brain volume for an atrophy disorder) unless you mean to.

**Discussion questions**
- For your own data, what would the reference cohort be, and what could bias it?
- When would you prefer z-scores over raw values as input to downstream machine learning?
- How would you communicate an individual's deviation map to a clinician?

**Key references**
- Marquand et al. (2016). Understanding heterogeneity in clinical cohorts using normative models. *Biological Psychiatry*.
- Marquand et al. (2019). Conceptualizing mental disorders as deviations from normative functioning. *Molecular Psychiatry*.
- Fraza et al. (2021). Warped Bayesian linear regression for normative modelling of big data. *NeuroImage*.
- Rutherford et al. (2022). The normative modeling framework for computational psychiatry. *Nature Protocols*.
- Rutherford et al. (2022). Charting brain growth and aging at high spatial precision. *eLife*.
- Bethlehem et al. (2022). Brain charts for the human lifespan. *Nature*.
- de Boer et al. (2024). Non-Gaussian normative modelling with hierarchical Bayesian regression. *Imaging Neuroscience*.
- PCNtoolkit documentation and tutorials: https://pcntoolkit.readthedocs.io